In [ ]:
import pandas as pd
import numpy as np
import re

# Helper functions
def clean_name(name):
    return re.sub(r'\s+', ' ', name).replace('.', '').strip().lower()

def convert_minutes_to_float(minutes_str):
    if isinstance(minutes_str, float):
        return minutes_str
    try:
        minutes, seconds = map(int, str(minutes_str).split(":"))
        return minutes + seconds / 60
    except Exception:
        return np.nan

# Main function
def prepare_defensive_features(data_path, hashmap_path, min_matchup_minutes=500):
    df = pd.read_csv(data_path)
    hashmap_df = pd.read_csv(hashmap_path)

    # Step 1: Normalize player names
    df['player_name'] = (
        df['first_name'].fillna("").astype(str) + " " +
        df['family_name'].fillna("").astype(str)
    ).map(clean_name)
    hashmap_df['player_name'] = hashmap_df['PLAYER'].astype(str).map(clean_name)

    # Step 2: Filter qualified defenders
    hashmap_df['GP'] = pd.to_numeric(hashmap_df['GP'], errors='coerce').fillna(0)
    hashmap_df['MIN'] = pd.to_numeric(hashmap_df['MIN'], errors='coerce').fillna(0)
    qualified_names = hashmap_df[(hashmap_df['MIN'] > 1300)]['player_name'].unique()
    df = df[df['player_name'].isin(qualified_names)].copy()

    # Step 3: Filter by matchup minutes
    if "matchup_minutes" in df.columns:
        df["matchup_minutes"] = df["matchup_minutes"].apply(convert_minutes_to_float)
        total_minutes = df.groupby("player_name")["matchup_minutes"].sum()
        qualified_by_minutes = total_minutes[total_minutes >= min_matchup_minutes].index
        df = df[df["player_name"].isin(qualified_by_minutes)].copy()
        print(f"Defenders with ≥{min_matchup_minutes} matchup minutes: {len(qualified_by_minutes)}")

    # Step 4: Convert numeric columns
    numeric_cols = [
        'matchup_field_goals_made', 'matchup_field_goals_attempted',
        'matchup_three_pointers_made', 'matchup_three_pointers_attempted',
        'matchup_free_throws_made', 'matchup_free_throws_attempted',
        'matchup_turnovers', 'matchup_blocks', 'help_blocks',
        'help_field_goals_made', 'help_field_goals_attempted',
        'shooting_fouls', 'matchup_minutes_sort', 'matchup_minutes', 'player_points'
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        else:
            print(f"⚠️ Warning: Column {col} missing.")

    # Step 5: Merge offensive rating data
    off = pd.read_csv("2024_offensive_advanced_metrics.csv")
    if 'Player' in off.columns:
        off['player_name'] = off['Player'].astype(str).map(clean_name)
    elif 'PLAYER' in off.columns:
        off['player_name'] = off['PLAYER'].astype(str).map(clean_name)
    else:
        raise ValueError("Offensive ratings file must contain a 'Player' or 'PLAYER' column with player names.")

    df['matchup_player'] = (
        df['matchups_first_name'].astype(str) + " " + df['matchups_family_name'].astype(str)
    ).map(clean_name)

    df = df.merge(off[['player_name', 'OFFRTG']], left_on='matchup_player', right_on='player_name', how='left')
    df = df.rename(columns={'OFFRTG': 'offensive_rating'})
    df = df[df['offensive_rating'].notna()].copy()  # drop matchups with no offensive rating

    # Step 6: Engineered Features
    df["minutes_played"] = df["matchup_minutes_sort"] / 60
    df["matchup_two_pointers_made"] = df["matchup_field_goals_made"] - df["matchup_three_pointers_made"]
    df["matchup_two_pointers_attempted"] = df["matchup_field_goals_attempted"] - df["matchup_three_pointers_attempted"]

    df["efg_allowed"] = np.where(
        df["matchup_field_goals_attempted"] > 0,
        (df["matchup_field_goals_made"] + 0.5 * df["matchup_three_pointers_made"]) / df["matchup_field_goals_attempted"],
        0
    )

    df["two_pt_allowed_pct"] = np.where(
        df["matchup_two_pointers_attempted"] > 0,
        df["matchup_two_pointers_made"] / df["matchup_two_pointers_attempted"],
        0
    )

    df["turnover_rate"] = np.where(
        df.get("partial_possessions", 0) > 0,
        df["matchup_turnovers"] / df["partial_possessions"],
        0
    )

    df["block_rate"] = np.where(
        df.get("partial_possessions", 0) > 0,
        df["matchup_blocks"] / df["partial_possessions"],
        0
    )

    df["help_stop_rate"] = np.where(
        df["help_field_goals_attempted"] > 0,
        1 - df["help_field_goals_made"] / df["help_field_goals_attempted"],
        0
    )

    df["coverage_share"] = df.get("percentage_defender_total_time", 0)
    df["points_allowed_per_100"] = np.where(
        df.get("partial_possessions", 0) > 0,
        df["player_points"] / df["partial_possessions"] * 100,
        0
    )

    df["fga_per_min"] = np.where(df["matchup_minutes"] > 0,
        df["matchup_field_goals_attempted"] / df["matchup_minutes"],
        0
    )

    df["3pa_per_min"] = np.where(df["matchup_minutes"] > 0,
        df["matchup_three_pointers_attempted"] / df["matchup_minutes"],
        0
    )

    df["fta_per_min"] = np.where(df["matchup_minutes"] > 0,
        df["matchup_free_throws_attempted"] / df["matchup_minutes"],
        0
    )

    df["switch_rate"] = np.where(
        df.get("matchup_minutes", 0) > 0,
        df.get("switches_on", 0) / df["matchup_minutes"],
        0
    )

    return df.reset_index(drop=True)

        
    

In [ ]:
features_from_2024_season_df = prepare_defensive_features(data_path="matchups_2024 2.csv", hashmap_path="2024_Player_Hashmap.csv")

# Check if 'first_name' exists in the DataFrame
if 'first_name' not in features_from_2024_season_df.columns:
	features_from_2024_season_df['first_name'] = None  # Add a placeholder column if missing

# Convert 'matchup_minutes_sort' to float and assign to 'matchup_minutes'
features_from_2024_season_df['matchup_minutes'] = features_from_2024_season_df['matchup_minutes_sort'].apply(convert_minutes_to_float)

features_from_2024_season_df

Defenders with ≥500 matchup minutes: 155


,game_id,away_team_id,home_team_id,team_id,team_name,team_city,team_tricode,team_slug,person_id,first_name,...,two_pt_allowed_pct,turnover_rate,block_rate,help_stop_rate,coverage_share,points_allowed_per_100,fga_per_min,3pa_per_min,fta_per_min,switch_rate
0,22400001,1610612737,1610612738,1610612738,Celtics,Boston,BOS,celtics,1627759,Jaylen,...,0.000000,0.000000,0.000000,0.0,0.083,0.000000,0.000000,0.000000,0.000000,0.0
1,22400001,1610612737,1610612738,1610612738,Celtics,Boston,BOS,celtics,1627759,Jaylen,...,0.750000,0.138889,0.000000,0.0,0.168,48.611111,1.355932,0.000000,0.677966,0.0
2,22400001,1610612737,1610612738,1610612738,Celtics,Boston,BOS,celtics,1627759,Jaylen,...,0.500000,0.078125,0.026042,0.0,0.472,52.083333,1.656051,0.636943,0.509554,0.0
3,22400001,1610612737,1610612738,1610612738,Celtics,Boston,BOS,celtics,1627759,Jaylen,...,1.000000,0.000000,0.000000,0.0,0.062,55.555556,1.463415,0.000000,0.000000,0.0
4,22400001,1610612737,1610612738,1610612738,Celtics,Boston,BOS,celtics,1627759,Jaylen,...,0.666667,0.000000,0.000000,0.0,0.064,93.023256,3.829787,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72700,22401230,1610612745,1610612760,1610612745,Rockets,Houston,HOU,rockets,1641708,Amen,...,0.600000,0.178571,0.000000,0.0,0.089,107.142857,4.225352,0.000000,0.000000,0.0
72701,22401230,1610612745,1610612760,1610612745,Rockets,Houston,HOU,rockets,1641708,Amen,...,0.500000,0.099010,0.099010,0.0,0.199,19.801980,1.411765,0.000000,0.000000,0.0
72702,22401230,1610612745,1610612760,1610612745,Rockets,Houston,HOU,rockets,1641708,Amen,...,0.000000,0.344828,0.000000,0.0,0.050,0.000000,0.000000,0.000000,0.000000,0.0
72703,22401230,1610612745,1610612760,1610612745,Rockets,Houston,HOU,rockets,1641708,Amen,...,0.000000,0.000000,0.000000,0.0,0.284,0.000000,0.895522,0.895522,0.000000,0.0


In [ ]:
%matplotlib inline

from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from IPython.display import display, HTML
import pandas as pd
import numpy as np

def derive_scoring_event(row): 
    help_fga = row.get("help_field_goals_attempted", 0)
    help_fgm = row.get("help_field_goals_made", 0)
    help_stop = (help_fga > 0 and help_fgm == 0)
    defensive_stop = (
        (row.get("matchup_blocks", 0) > 0) |
        (row.get("help_blocks", 0) > 0) |
        (help_stop) |
        (row.get("matchup_minutes", 0) < 0.5) |       # short stints
        (row.get("coverage_share", 0) > 0.9)          # great effort
    )   
    return 0 if defensive_stop else 1

def select_safe_features(df):
    return df[[
        'matchup_minutes', 'fga_per_min', '3pa_per_min',
        'fta_per_min', 'switch_rate', 'coverage_share'
    ]].copy()

def train_defensive_forest_balanced(df, test_size=0.2, random_state=42):
    df['scoring_event'] = df.apply(derive_scoring_event, axis=1)
    
    # Balance classes via upsampling
    majority = df[df['scoring_event'] == 1]
    minority = df[df['scoring_event'] == 0]
    if len(minority) > 0:
        minority_upsampled = resample(minority, replace=True, n_samples=len(majority), random_state=random_state)
        df_balanced = pd.concat([majority, minority_upsampled])
    else:
        df_balanced = df.copy()

    # Features and labels
    X = select_safe_features(df_balanced)
    y = df_balanced['scoring_event']

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )

    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        min_samples_leaf=4,
        min_samples_split=10,
        max_features='sqrt',
        class_weight=None,  
        random_state=random_state,
        n_jobs=-1
    )
    

    model.fit(X_train, y_train)
    calibrated_model = CalibratedClassifierCV(model, method='isotonic', cv=5)
    calibrated_model.fit(X_train, y_train)

    # Evaluate
    y_pred_proba = calibrated_model.predict_proba(X_test)[:, 1]
    print("\nModel Evaluation:")
    print(f"Log Loss: {log_loss(y_test, y_pred_proba):.4f}")
    print(f"AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")

    # Score all defenders
    full_X = select_safe_features(df)
    df['scoring_probability'] = calibrated_model.predict_proba(full_X)[:, 1]

    defender_scores_df = (
        df.groupby(['person_id', 'first_name', 'family_name', 'position'])['scoring_probability']
        .mean()
        .reset_index()
        .rename(columns={'scoring_probability': 'avg_scoring_probability'})
        .sort_values(by='avg_scoring_probability')
    )
    defender_scores_df = defender_scores_df.merge(
        df.groupby('person_id').size().reset_index(name='possessions'),
        on='person_id', how='left'
    )

    print("\nTop Defenders by Average Scoring Probability:")
    display(HTML(defender_scores_df.head(100).to_html(index=False)))

    return calibrated_model, defender_scores_df, X_test, y_test

# Run the model
model, defender_scores_df, X_test, y_test = train_defensive_forest_balanced(features_from_2024_season_df)
print(defender_scores_df['avg_scoring_probability'].describe())



Model Evaluation:
Log Loss: 0.4798
AUC Score: 0.8190

Top Defenders by Average Scoring Probability:


person_id,first_name,family_name,position,avg_scoring_probability,possessions
1628983,Shai,Gilgeous-Alexander,G,0.439267,535
1629628,RJ,Barrett,G,0.441695,398
1631097,Bennedict,Mathurin,G,0.454838,476
1631094,Paolo,Banchero,F,0.457953,313
1630578,Alperen,Sengun,F,0.466695,525
1631114,Jalen,Williams,C,0.469169,483
1630532,Franz,Wagner,F,0.472398,396
203507,Giannis,Antetokounmpo,F,0.477897,459
1630178,Tyrese,Maxey,G,0.486418,373
202681,Kyrie,Irving,G,0.493101,359


count    212.000000
mean       0.640836
std        0.094705
min        0.439267
25%        0.568174
50%        0.632336
75%        0.710087
max        0.900260
Name: avg_scoring_probability, dtype: float64
